# 04 - Feature Engineering

Separação de features:
- **X_regime**: Features macro para GMM
- **X_signal**: Features direcionais para XGBoost

In [ ]:
import pandas as pd
import numpy as np
import json
import pickle
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import sys
sys.path.append('../src')
from fracdiff import fdiff

DATA_DIR = Path('../data')

In [ ]:
df = pd.read_parquet(DATA_DIR / 'signals_labeled.parquet')
print(f'Total: {len(df)}')

## 1. Carregar d Ótimos

In [ ]:
try:
    with open(DATA_DIR / 'optimal_d.json', 'r') as f:
        optimal_d = json.load(f)
    print('d ótimos:', optimal_d)
except:
    print('Usando d=0.4')
    optimal_d = {'f_price_context': 0.4}

## 2. X_regime (GMM) - Com StandardScaler

In [ ]:
regime_features = ['f_volume', 'f_momentum', 'f_trend', 'hour_utc']
X_regime = df[regime_features].copy()

scaler_regime = StandardScaler()
X_regime_scaled = pd.DataFrame(
    scaler_regime.fit_transform(X_regime),
    columns=regime_features
)
print('X_regime:', X_regime_scaled.shape)

## 3. X_signal (XGBoost) - FracDiff só em preço

In [ ]:
# FracDiff APENAS em f_price_context
d = optimal_d.get('f_price_context', 0.4)
df['f_price_context_frac'] = fdiff(df['f_price_context'].values, d)

# Features direcionais (discretas SEM FracDiff)
signal_features = [
    'f_price_context_frac',  # FracDiff
    'f_smart_money',          # Discreto
    'f_imbalance',            # Discreto
    'f_absorption',           # Discreto
    'score',
]

X_signal = df[signal_features].fillna(0)
print('X_signal:', X_signal.shape)

In [ ]:
y = df['y'].values
print(f'Target: {y.sum()} WIN / {(~y.astype(bool)).sum()} LOSS')

## 4. Salvar

In [ ]:
features_dict = {
    'X_regime': X_regime_scaled,
    'X_signal': X_signal,
    'y': y,
    'scaler_regime': scaler_regime,
    'regime_features': regime_features,
    'signal_features': signal_features,
}

with open(DATA_DIR / 'features.pkl', 'wb') as f:
    pickle.dump(features_dict, f)
print('Salvo!')